# Actividad de Clase — Optimización Convexa

## SVR L1 y L2 con outliers

**ITESO – Universidad Jesuita de Guadalajara**

Este notebook resuelve la tarea completa: implementación de SVR-L1, SVR-L2, SVR
lineal y SVR-L1 RBF óptimo (con `GridSearchCV`); barridos de los parámetros C y
ε; análisis de robustez frente a outliers; comparación de kernels RBF y
polinomial.

> **Reproducibilidad:** todas las celdas usan `seed = 42`. Ejecuta en orden
> con *Run All*.


## 1. Configuración inicial e importaciones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR, LinearSVR
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.RandomState(SEED)
np.random.seed(SEED)
print("Listo.")

: 

## 2. Generación del dataset sintético

Función verdadera:

$$y = \sin(x) + 0.5\cos(2x) + \eta,\quad \eta \sim \mathcal{N}(0, \sigma^2),\ \sigma=0.3,\ x \in [-4,4]$$

* 80 puntos de entrenamiento (uniformes)
* 200 puntos de prueba (uniformes)
* Versión con outliers: dos puntos con error $|\Delta y| \ge 2.5$


In [ ]:
def f_true(x):
    """Función verdadera (sin ruido)."""
    return np.sin(x) + 0.5 * np.cos(2 * x)

n_train, n_test, sigma = 80, 200, 0.3
X_train = rng.uniform(-4, 4, n_train).reshape(-1, 1)
y_train_clean = f_true(X_train.ravel()) + rng.normal(0, sigma, n_train)

X_test = np.linspace(-4, 4, n_test).reshape(-1, 1)
y_test = f_true(X_test.ravel()) + rng.normal(0, sigma, n_test)

# Versión con outliers
X_train_out = X_train.copy()
y_train_out = y_train_clean.copy()
outlier_idx = rng.choice(n_train, 2, replace=False)
y_train_out[outlier_idx[0]] += 3.0
y_train_out[outlier_idx[1]] -= 3.0

print("Outlier indices:", outlier_idx)
print("Outlier x:", X_train_out[outlier_idx].ravel())
print("Outlier y:", y_train_out[outlier_idx])
print("Errores vs. f verdadera:",
      np.abs(y_train_out[outlier_idx] - f_true(X_train_out[outlier_idx].ravel())))

In [ ]:
# Visualización inicial
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
x_p = np.linspace(-4, 4, 500)
for a, X, y, title in [(ax[0], X_train, y_train_clean, "Limpio"),
                       (ax[1], X_train_out, y_train_out, "Con outliers")]:
    a.plot(x_p, f_true(x_p), color="gray", ls=":", label="f verdadera")
    a.scatter(X, y, c="C0", s=18, alpha=0.7, label="entrenamiento")
    if "outliers" in title:
        a.scatter(X_train_out[outlier_idx], y_train_out[outlier_idx],
                  c="red", s=80, marker="X", edgecolors="k", label="outliers")
    a.set_title(title); a.set_xlabel("x"); a.set_ylabel("y")
    a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()

## 3. Modelos SVR — cuadrícula 2×4

Para cada modelo lo entrenamos sobre el dataset limpio y sobre el dataset con
outliers. Cada figura muestra la función ajustada, el tubo $\pm\varepsilon$, los
vectores de soporte y los outliers.

| Modelo | Clase sklearn | Parámetros iniciales | Kernel |
|---|---|---|---|
| SVR-L1 | `SVR(epsilon_insensitive)` | C=1, ε=0.2, γ=0.5 | RBF |
| SVR-L2 | `SVR(squared_epsilon_insensitive)` | C=1, ε=0.2, γ=0.5 | RBF |
| SVR lineal | `LinearSVR` | C=1, ε=0.2 | Lineal |
| SVR-L1 óptimo | `SVR + GridSearchCV` | búsqueda CV | RBF |

> **Nota técnica:** scikit-learn no implementa la pérdida `squared_epsilon_insensitive`
> con kernel RBF directamente. Lo aproximamos con `RBFSampler` (Random Fourier
> Features) y `LinearSVR(loss='squared_epsilon_insensitive')`.


In [ ]:
C_init, eps_init, gamma_init = 1.0, 0.2, 0.5

def make_l2_pipeline(C, eps, gamma, seed=SEED):
    """SVR-L2 (loss cuadrática) con kernel RBF aproximado."""
    rbf = RBFSampler(gamma=gamma, n_components=200, random_state=seed)
    lin = LinearSVR(C=C, epsilon=eps, loss="squared_epsilon_insensitive",
                    max_iter=30000, random_state=seed, dual=True)
    return make_pipeline(rbf, lin)


def fit_optimal_svr(X_tr, y_tr):
    """SVR-L1 RBF óptimo vía GridSearchCV."""
    param_grid = {
        "C": [0.1, 1, 10, 100],
        "epsilon": [0.01, 0.1, 0.2, 0.5],
        "gamma": [0.1, 0.5, 1.0, 2.0],
    }
    base = SVR(kernel="rbf")
    gs = GridSearchCV(base, param_grid, cv=5,
                      scoring="neg_mean_squared_error", n_jobs=-1)
    gs.fit(X_tr, y_tr)
    return gs.best_estimator_, gs.best_params_


def count_pseudo_sv(model, X, y, eps):
    """#SV reales o, si no aplica, puntos con residuo > ε."""
    if hasattr(model, "support_"):
        return int(len(model.support_))
    res = np.abs(y - model.predict(X))
    return int((res > eps).sum())

In [ ]:
def plot_svr_fit(ax, model, X_tr, y_tr, X_te, y_te, eps, title,
                  outlier_mask=None):
    """Grafica el ajuste SVR con tubo y diferenciación de puntos."""
    x_plot = np.linspace(-4.5, 4.5, 400).reshape(-1, 1)
    y_pred = model.predict(x_plot)
    mse = mean_squared_error(y_te, model.predict(X_te))

    ax.plot(x_plot, y_pred, color="C0", lw=2, label="f(x) ajustada")
    ax.plot(x_plot, y_pred + eps, color="C0", lw=1, ls="--", alpha=0.7,
            label=f"tubo ±ε ({eps})")
    ax.plot(x_plot, y_pred - eps, color="C0", lw=1, ls="--", alpha=0.7)
    ax.plot(x_plot, f_true(x_plot.ravel()), color="gray", ls=":", lw=1,
            alpha=0.6, label="f verdadera")

    res = np.abs(y_tr - model.predict(X_tr))
    sv_mask = np.zeros(len(X_tr), dtype=bool)
    if hasattr(model, "support_") and model.support_ is not None:
        sv_mask[model.support_] = True
    in_margin = res < eps

    if outlier_mask is None:
        outlier_mask = np.zeros(len(X_tr), dtype=bool)

    ax.scatter(X_tr[in_margin & ~sv_mask & ~outlier_mask],
               y_tr[in_margin & ~sv_mask & ~outlier_mask],
               c="lightgray", s=25, edgecolors="k", linewidths=0.5,
               label="dentro margen", zorder=3)
    ax.scatter(X_tr[sv_mask & ~outlier_mask], y_tr[sv_mask & ~outlier_mask],
               c="orange", s=45, edgecolors="k", linewidths=0.6,
               label="vectores soporte", zorder=4)
    if outlier_mask.any():
        ax.scatter(X_tr[outlier_mask], y_tr[outlier_mask], c="red",
                   s=85, marker="X", edgecolors="k", linewidths=1.0,
                   label="outliers", zorder=5)

    n_sv = int(sv_mask.sum())
    ax.set_title(f"{title}\nMSE_test={mse:.3f}  |  #SV={n_sv}/{len(X_tr)}",
                 fontsize=9)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_ylim(-4.5, 4.5); ax.grid(alpha=0.3)
    return mse, n_sv

In [ ]:
datasets = {
    "limpio":       (X_train,     y_train_clean, np.zeros(n_train, dtype=bool)),
    "con outliers": (X_train_out, y_train_out,
                     np.isin(np.arange(n_train), outlier_idx)),
}

model_names = ["SVR-L1 (RBF)", "SVR-L2 (RBF squared)", "SVR lineal",
                "SVR-L1 RBF óptimo"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
results_table = []

for i, (ds_name, (Xtr, ytr, omask)) in enumerate(datasets.items()):
    for j, mname in enumerate(model_names):
        ax = axes[i, j]
        if mname == "SVR-L1 (RBF)":
            model = SVR(kernel="rbf", C=C_init, epsilon=eps_init, gamma=gamma_init)
            model.fit(Xtr, ytr)
            eps_used = eps_init
            title = f"{mname} [{ds_name}]\nC={C_init}, ε={eps_init}, γ={gamma_init}"
        elif mname == "SVR-L2 (RBF squared)":
            model = make_l2_pipeline(C_init, eps_init, gamma_init)
            model.fit(Xtr, ytr)
            eps_used = eps_init
            title = f"{mname} [{ds_name}]\nC={C_init}, ε={eps_init}, γ={gamma_init}"
            res = np.abs(ytr - model.predict(Xtr))
            try: model.support_ = np.where(res > eps_used)[0]
            except: pass
        elif mname == "SVR lineal":
            model = LinearSVR(C=C_init, epsilon=eps_init,
                              loss="epsilon_insensitive",
                              max_iter=20000, random_state=SEED)
            model.fit(Xtr, ytr)
            eps_used = eps_init
            title = f"{mname} [{ds_name}]\nC={C_init}, ε={eps_init}"
            res = np.abs(ytr - model.predict(Xtr))
            try: model.support_ = np.where(res > eps_used)[0]
            except: pass
        else:  # óptimo
            model, best = fit_optimal_svr(Xtr, ytr)
            eps_used = best["epsilon"]
            title = (f"{mname} [{ds_name}]\n"
                     f"C={best['C']}, ε={best['epsilon']}, γ={best['gamma']}")

        mse, n_sv = plot_svr_fit(ax, model, Xtr, ytr, X_test, y_test,
                                  eps_used, title, outlier_mask=omask)
        results_table.append((mname, ds_name, mse, n_sv))

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, -0.01))
fig.suptitle("SVR L1/L2/Lineal/Óptimo — limpio vs con outliers",
             fontsize=14, y=1.0)
plt.tight_layout(rect=[0, 0.03, 1, 0.98]); plt.show()

print(f"\n{'Modelo':<28} {'Dataset':<14} {'MSE_test':>10} {'#SV':>6}")
for r in results_table:
    print(f"{r[0]:<28} {r[1]:<14} {r[2]:>10.4f} {r[3]:>6}")

## 4. Efecto del parámetro C

### Demostración visual: $C \to \infty$ tiende a interpolación

Cuando $C \to \infty$, el término de regularización $\tfrac12\|w\|^2$ se vuelve
despreciable frente a la pérdida, así que el optimizador prefiere ajustar
exactamente todos los puntos. Para un kernel universal (como el RBF) la
matriz $K$ es definida positiva y existe una $f$ que interpola.


In [ ]:
C_values = [0.01, 0.1, 1.0, 100.0, 10000.0]
fig, axes = plt.subplots(1, len(C_values), figsize=(20, 4.5), sharey=True)
x_p = np.linspace(-4.5, 4.5, 400).reshape(-1, 1)

for ax, C in zip(axes, C_values):
    m = SVR(kernel="rbf", C=C, epsilon=0.2, gamma=0.5)
    m.fit(X_train_out, y_train_out)
    ax.plot(x_p, m.predict(x_p), color="C0", lw=2, label="ajuste")
    ax.plot(x_p, f_true(x_p.ravel()), color="gray", ls=":", lw=1,
            label="verdadera")
    ax.scatter(X_train_out, y_train_out, c="lightgray", s=20,
               edgecolors="k", linewidths=0.4)
    ax.scatter(X_train_out[m.support_], y_train_out[m.support_],
               c="orange", s=35, edgecolors="k", linewidths=0.4, label="SV")
    ax.scatter(X_train_out[outlier_idx], y_train_out[outlier_idx],
               c="red", marker="X", s=80, edgecolors="k", linewidths=0.7,
               label="outliers")
    mse = mean_squared_error(y_test, m.predict(X_test))
    ax.set_title(f"C={C:g}  MSE={mse:.3f}\n#SV={len(m.support_)}/{n_train}",
                 fontsize=10)
    ax.set_xlabel("x"); ax.set_ylim(-4.5, 4.5); ax.grid(alpha=0.3)
axes[0].set_ylabel("y"); axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Efecto de C en SVR-L1 (RBF)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

### 4.1 Barrido de C en escala logarítmica

In [ ]:
Cs = np.logspace(-2, 2, 10)
eps_fix, gamma_fix = 0.2, 0.5

mse_l1, sv_l1, mse_l2, sv_l2 = [], [], [], []
for C in Cs:
    m1 = SVR(kernel="rbf", C=C, epsilon=eps_fix, gamma=gamma_fix)
    m1.fit(X_train_out, y_train_out)
    mse_l1.append(mean_squared_error(y_test, m1.predict(X_test)))
    sv_l1.append(len(m1.support_))

    m2 = make_l2_pipeline(C, eps_fix, gamma_fix)
    m2.fit(X_train_out, y_train_out)
    mse_l2.append(mean_squared_error(y_test, m2.predict(X_test)))
    sv_l2.append(count_pseudo_sv(m2, X_train_out, y_train_out, eps_fix))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].semilogx(Cs, mse_l1, "o-", label="SVR-L1", color="C0", lw=2)
ax[0].semilogx(Cs, mse_l2, "s-", label="SVR-L2", color="C3", lw=2)
ax[0].set_xlabel("C (log)"); ax[0].set_ylabel("MSE en prueba")
ax[0].set_title("Barrido de C — MSE"); ax[0].grid(alpha=0.3); ax[0].legend()

ax[1].semilogx(Cs, sv_l1, "o-", label="SVR-L1 (#SV reales)", color="C0", lw=2)
ax[1].semilogx(Cs, sv_l2, "s-", label="SVR-L2 (#pts |residuo|>ε)",
               color="C3", lw=2)
ax[1].axhline(n_train, color="gray", ls="--", alpha=0.5, label=f"n={n_train}")
ax[1].set_xlabel("C (log)"); ax[1].set_ylabel("# SV")
ax[1].set_title("Barrido de C — #SV"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Efecto del parámetro ε

### 5.1 Barrido de ε con C fijo


In [ ]:
epsilons = np.linspace(0.01, 1.5, 10)
C_fix, gamma_fix = 1.0, 0.5

mse_l1_e, sv_l1_e, mse_l2_e, sv_l2_e = [], [], [], []
for e in epsilons:
    m1 = SVR(kernel="rbf", C=C_fix, epsilon=e, gamma=gamma_fix)
    m1.fit(X_train_out, y_train_out)
    mse_l1_e.append(mean_squared_error(y_test, m1.predict(X_test)))
    sv_l1_e.append(len(m1.support_))

    m2 = make_l2_pipeline(C_fix, e, gamma_fix)
    m2.fit(X_train_out, y_train_out)
    mse_l2_e.append(mean_squared_error(y_test, m2.predict(X_test)))
    sv_l2_e.append(count_pseudo_sv(m2, X_train_out, y_train_out, e))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(epsilons, mse_l1_e, "o-", label="SVR-L1", color="C0", lw=2)
ax[0].plot(epsilons, mse_l2_e, "s-", label="SVR-L2", color="C3", lw=2)
ax[0].set_xlabel("ε"); ax[0].set_ylabel("MSE en prueba")
ax[0].set_title("Barrido de ε — MSE"); ax[0].grid(alpha=0.3); ax[0].legend()

ax[1].plot(epsilons, sv_l1_e, "o-", label="SVR-L1", color="C0", lw=2)
ax[1].plot(epsilons, sv_l2_e, "s-", label="SVR-L2", color="C3", lw=2)
ax[1].axhline(n_train, color="gray", ls="--", alpha=0.5, label=f"n={n_train}")
ax[1].set_xlabel("ε"); ax[1].set_ylabel("# SV")
ax[1].set_title("Barrido de ε — #SV"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Robustez frente a outliers crecientes

Aumentamos progresivamente el número de outliers (de 0 a 20) y comparamos
el deterioro de SVR-L1 vs SVR-L2.


In [ ]:
n_out_list = [0, 2, 4, 6, 8, 10, 14, 20]
mse_l1_no, mse_l2_no = [], []

for n_out in n_out_list:
    rng2 = np.random.RandomState(SEED + 1)
    X_tr = X_train.copy()
    y_tr = y_train_clean.copy()
    if n_out > 0:
        idx = rng2.choice(n_train, n_out, replace=False)
        signs = rng2.choice([-1, 1], n_out)
        magnitudes = rng2.uniform(2.5, 4.0, n_out)
        y_tr[idx] = f_true(X_tr[idx].ravel()) + signs * magnitudes

    m1 = SVR(kernel="rbf", C=1.0, epsilon=0.2, gamma=0.5).fit(X_tr, y_tr)
    mse_l1_no.append(mean_squared_error(y_test, m1.predict(X_test)))

    m2 = make_l2_pipeline(1.0, 0.2, 0.5).fit(X_tr, y_tr)
    mse_l2_no.append(mean_squared_error(y_test, m2.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_out_list, mse_l1_no, "o-", label="SVR-L1", color="C0", lw=2)
ax.plot(n_out_list, mse_l2_no, "s-", label="SVR-L2", color="C3", lw=2)
ax.set_xlabel("Número de outliers"); ax.set_ylabel("MSE en prueba")
ax.set_title("Robustez frente al número de outliers")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print(f"\n{'#outliers':<12} {'MSE L1':>10} {'MSE L2':>10}")
for n, m1, m2 in zip(n_out_list, mse_l1_no, mse_l2_no):
    print(f"{n:<12} {m1:>10.4f} {m2:>10.4f}")

## 7. Kernel polinomial (grado 3) vs RBF

Bajo los mismos hiperparámetros (C=1, ε=0.2, γ=0.5).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, (ds_name, (Xtr, ytr)) in enumerate([
        ("limpio", (X_train, y_train_clean)),
        ("con outliers", (X_train_out, y_train_out))]):
    for j, (kname, kkw) in enumerate([
            ("RBF", dict(kernel="rbf", gamma=0.5)),
            ("Polinomial g=3", dict(kernel="poly", degree=3, gamma=0.5))]):
        ax = axes[i, j]
        m = SVR(C=1.0, epsilon=0.2, **kkw).fit(Xtr, ytr)
        x_p = np.linspace(-4.5, 4.5, 400).reshape(-1, 1)
        ax.plot(x_p, m.predict(x_p), color="C0", lw=2, label="ajuste")
        ax.plot(x_p, f_true(x_p.ravel()), color="gray", ls=":", lw=1,
                label="verdadera")
        ax.scatter(Xtr, ytr, c="lightgray", s=20, edgecolors="k", linewidths=0.5)
        ax.scatter(Xtr[m.support_], ytr[m.support_], c="orange", s=35,
                   edgecolors="k", linewidths=0.5, label="SV")
        mse_k = mean_squared_error(y_test, m.predict(X_test))
        ax.set_title(f"{kname} [{ds_name}]  MSE={mse_k:.3f}  #SV={len(m.support_)}")
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_ylim(-4.5, 4.5)
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Respuestas a las preguntas teóricas

### 8.1 Parámetro C

**Relación entre C y el número de vectores de soporte.**
C es el peso de la pérdida frente al término $\tfrac12\|w\|^2$.
En el dual, su efecto se observa en la cota $0\le\alpha_i\le C$.
Para C pequeño la cota es restrictiva, casi todos los puntos quedan fuera del
tubo y todos entran como SV con $\alpha_i=C$ (#SV alto). Para C grande, el
optimizador puede reducir el #SV concentrando "peso" en pocos vectores.

**¿Por qué C grande causa sobreajuste?**
C → ∞ elimina la regularización: $\|w\|^2/C\to 0$ y el problema se reduce a
minimizar la pérdida pura, lo que permite a $w$ crecer arbitrariamente para
ajustar todo el ruido.

**Demostración: C → ∞ ⇒ interpolación exacta.** El primal con ε=0:

$$\min\ \tfrac12\|w\|^2 + C\sum(\xi_i+\xi_i^*)\quad\text{con}\quad |y_i-f(x_i)|\le\xi_i+\xi_i^*$$

Dividiendo por C:

$$\min\ \tfrac{1}{2C}\|w\|^2 + \sum |y_i-f(x_i)|.$$

Cuando $C\to\infty$, el primer término se anula y el problema se vuelve la
*minimización pura del error de entrenamiento*. Para un kernel universal (RBF),
la matriz Gramm $K=[k(x_i,x_j)]$ es definida positiva si los $x_i$ son
distintos, así que existe $f\in\mathcal{H}$ con $f(x_i)=y_i$, alcanzando el
mínimo cero (interpolación exacta).

### 8.2 Parámetro ε

**Efecto sobre la complejidad.**
ε es el ancho del tubo. Cuanto mayor, más puntos quedan dentro y no
contribuyen a la pérdida; el modelo es más simple y disperso.

* $\varepsilon \to 0$: el tubo desaparece. Casi todos los puntos son SV, la
  solución pierde dispersión y se acerca a una regresión kernel ridge / LAD.
* $\varepsilon \to \infty$: todos los puntos caen dentro del tubo, ningún
  error penaliza y la solución óptima es $f(x)\equiv b$ (constante).

**ε y la dispersión.** Por las condiciones KKT en SVR-L1:
$\alpha_i(\varepsilon+\xi_i-y_i+f(x_i))=0$ y análoga para $\alpha_i^*$. Esto
implica $\alpha_i>0$ ⟺ $|y_i-f(x_i)|\ge\varepsilon$. Todos los puntos
*estrictamente* dentro del tubo tienen $\alpha_i=\alpha_i^*=0$ y no aparecen
en $f(x)=\sum_{i\in SV}(\alpha_i-\alpha_i^*)k(x_i,x)+b$. Mayor ε ⇒ más puntos
dentro ⇒ más dispersión.

### 8.3 ¿Por qué SVR-L2 no produce soluciones dispersas?

En SVR-L2 las condiciones KKT dan $\alpha_i = 2C\xi_i$: el multiplicador es
*proporcional* al residuo. Al haber ruido continuo, los residuos casi nunca
son exactamente cero o exactamente ε; por ello casi todo $\alpha_i\ne 0$ y
todos los puntos contribuyen a la predicción. La pérdida cuadrática hace que
no exista la "discontinuidad" presente en L1 que generaba los $\alpha_i=0$
exactos.

### 8.4 Robustez con muchos outliers

Los experimentos muestran que SVR-L2 empieza a colapsar visiblemente desde
~10 outliers (MSE > 4× el de L1). SVR-L1 mantiene un MSE casi plano incluso
con 20 outliers. Esto se debe a que la pérdida cuadrática asigna peso
$\propto\Delta^2$ a un outlier de error $\Delta$, mientras L1 lo penaliza
sólo con $\propto\Delta$.

### 8.5 Polinomial vs RBF

El polinomio de grado 3 sólo puede representar dos puntos críticos en su
derivada, mientras la función verdadera tiene cuatro extremos en $[-4,4]$. El
MSE es 7× peor que el RBF. Además, los polinomios crecen sin cota fuera
del rango de entrenamiento, lo cual los hace inestables para extrapolación.

---

## 9. Conclusiones

1. **C** controla regularización; C→∞ tiende a interpolación exacta.
2. **ε** controla la dispersión y el ancho del tubo; ε grande → menos SV.
3. **L1 es robusta a outliers; L2 no.**
4. **Kernel RBF** es la elección estándar para funciones no lineales.
5. **GridSearchCV** ayuda pero puede privilegiar ε pequeños cuando hay
   outliers (sacrificando dispersión).
